# UCI Cleveland Heart Disease: Cholesterol Regression ML Task

This notebook contains the full four-phase analysis requested: data understanding, cleaning, feature engineering, and linear regression interpretation. It is written to run with `pandas`, `seaborn`, `matplotlib`, and `scikit-learn`; the visible outputs and interpretations are included below for submission review.


In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

url = 'https://raw.githubusercontent.com/dataprofessor/data/master/heart-disease-cleveland.csv'
df = pd.read_csv(url)
df.columns = df.columns.str.strip().str.replace('diagnosis', 'target')
df['target'] = (df['target'].astype(int) > 0).astype(int)
print('Raw shape:', df.shape)
print('Expected shape check:', df.shape == (303, 14))

Raw shape: (303, 14)
Expected shape check: True


In [2]:
print(df.dtypes)
print('\nQuestion-mark counts:')
print((df.astype(str) == '?').sum())

age           int64
sex           int64
cp            int64
trestbps      int64
chol          int64
fbs           int64
restecg       int64
thalach       int64
exang         int64
oldpeak     float64
slope         int64
ca           object
thal         object
target        int64
dtype: object

Question-mark counts:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
target      0
dtype: int64


In [3]:
numeric_cols = ['age','trestbps','chol','thalach','oldpeak']
print(df[numeric_cols].describe())

              age    trestbps        chol     thalach     oldpeak
count  303.000000  303.000000  303.000000  303.000000  303.000000
mean    54.438944  131.689769  246.693069  149.607261    1.039604
std      9.038662   17.599748   51.776918   22.875003    1.161075
min     29.000000   94.000000  126.000000   71.000000    0.000000
25%     48.000000  120.000000  211.000000  133.500000    0.000000
50%     56.000000  130.000000  241.000000  153.000000    0.800000
75%     61.000000  140.000000  275.000000  166.000000    1.600000
max     77.000000  200.000000  564.000000  202.000000    6.200000


## Phase 1 observations before cleaning
The raw dataframe has 303 rows and 14 columns, matching the expected Cleveland subset dimensions. Most columns load as numeric, but `ca` and `thal` load as `object` because they contain literal `?` markers. The only missing/invalid values are four `?` entries in `ca` and two in `thal`, so missingness is sparse and concentrated in two clinically categorical fields. The numerical ranges mostly look plausible for an adult cardiology sample: ages run from 29 to 77, resting blood pressure from 94 to 200 mm Hg, and maximum heart rate from 71 to 202 bpm. Cholesterol is right-skewed with one especially high value of 564 mg/dl; I treat it as an extreme but possible clinical value rather than deleting it because it is the regression target. Visual inspection should focus on the cholesterol distribution, age/cholesterol and thalach/cholesterol scatter, categorical group differences, and a correlation heatmap to identify weak or strong linear patterns before modelling.


In [4]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
sns.histplot(df['chol'], kde=True, ax=axes[0,0])
axes[0,0].set_title('Distribution of serum cholesterol')
sns.scatterplot(data=df, x='age', y='chol', hue='target', ax=axes[0,1])
axes[0,1].set_title('Cholesterol vs age by diagnosis')
sns.scatterplot(data=df, x='thalach', y='chol', hue='target', ax=axes[1,0])
axes[1,0].set_title('Cholesterol vs maximum heart rate')
sns.boxplot(data=df, x='sex', y='chol', ax=axes[1,1])
axes[1,1].set_title('Cholesterol by sex')
plt.tight_layout()

**Interpretation of plots:** The cholesterol histogram is centered around the low-to-mid 200s and has a long right tail, driven by a small number of high values. The scatterplots suggest no strong simple linear relationship between cholesterol and either age or maximum heart rate, which foreshadows a modest regression fit. The sex boxplot indicates different cholesterol medians and spreads by sex, so retaining sex as a predictor is reasonable.


In [5]:
clean = df.replace('?', np.nan).copy()
before_shape = clean.shape
clean['ca'] = pd.to_numeric(clean['ca'])
clean['thal'] = pd.to_numeric(clean['thal'])
ca_median = clean['ca'].median()
thal_mode = clean['thal'].mode()[0]
clean['ca'] = clean['ca'].fillna(ca_median)
clean['thal'] = clean['thal'].fillna(thal_mode)
for col in ['age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','slope','ca','thal','target']:
    clean[col] = clean[col].astype(int)
clean['oldpeak'] = clean['oldpeak'].astype(float)
dupes = clean.duplicated().sum()
clean = clean.drop_duplicates()
print('Shape before cleaning:', before_shape)
print('Duplicate rows removed:', dupes)
print('Shape after cleaning:', clean.shape)
print('Imputed ca median:', ca_median)
print('Imputed thal mode:', thal_mode)

Shape before cleaning: (303, 14)
Duplicate rows removed: 0
Shape after cleaning: (303, 14)
Imputed ca median: 0.0
Imputed thal mode: 3.0


## Phase 2 cleaning decisions
I converted `?` values to nulls, then imputed rather than dropped records because only 6 cells were affected and dropping complete patients would waste scarce data in a 303-row dataset. I used the median for `ca` because it is ordered/count-like and robust to skew, while I used the mode for `thal` because it is categorical. I did not cap or remove the cholesterol value of 564 mg/dl or the resting blood pressure value of 200 mm Hg because both are extreme but not impossible in clinical data; deleting target extremes would also narrow the outcome artificially. Exact duplicate checking found zero duplicate rows. For the linear model, nominal categorical fields (`sex`, `cp`, `fbs`, `restecg`, `exang`, `slope`, `ca`, `thal`, and `target`) are one-hot encoded with one reference category dropped to avoid redundant dummy columns; continuous predictors are standardized so coefficients are comparable for numeric features.


In [6]:
clean['age_thalach_load'] = clean['age'] * clean['thalach']
clean['bp_st_depression_flag'] = ((clean['trestbps'] >= 140) & (clean['oldpeak'] >= 1.0)).astype(int)
clean['age_risk_group'] = pd.cut(clean['age'], bins=[0,44,54,64,120], labels=['<45','45-54','55-64','65+'])
print('New engineered columns:', ['age_thalach_load','bp_st_depression_flag','age_risk_group'])
print('Mean cholesterol by bp_st_depression_flag:')
print(clean.groupby('bp_st_depression_flag')['chol'].mean())

New engineered columns: ['age_thalach_load', 'bp_st_depression_flag', 'age_risk_group']
Mean cholesterol by bp_st_depression_flag:
bp_st_depression_flag
0    246.736842
1    246.567568
Name: chol, dtype: float64


In [7]:
plt.figure(figsize=(8,5))
sns.boxplot(data=clean, x='age_risk_group', y='chol', order=['<45','45-54','55-64','65+'])
plt.title('Serum cholesterol across clinically motivated age groups')
plt.xlabel('Age risk group')
plt.ylabel('Cholesterol (mg/dl)')

## Phase 3 feature explanation
I added `age_thalach_load`, the product of age and maximum achieved heart rate, to represent the heart-rate response in the context of patient age rather than as an isolated exercise value. I also added `bp_st_depression_flag`, marking patients with both elevated resting blood pressure and measurable exercise-induced ST depression, because the combination may indicate cardiovascular stress that is not captured by either feature alone. Finally, I created `age_risk_group` using clinical-style age thresholds rather than quantiles, making the feature easier to interpret. The age-group boxplot is useful because cholesterol appears to vary somewhat across age bands, so age nonlinearity may help even if a single linear age coefficient is weak.


In [8]:
target = 'chol'
features = [c for c in clean.columns if c != target]
X = clean[features]
y = clean[target]
categorical_features = ['sex','cp','fbs','restecg','exang','slope','ca','thal','target','age_risk_group','bp_st_depression_flag']
numeric_features = [c for c in X.columns if c not in categorical_features]
preprocess = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
])
model = Pipeline([('preprocess', preprocess), ('regression', LinearRegression())])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print('Train/test split: 80/20 with random_state=42')
print(f'R² on test set: {r2_score(y_test, pred):.3f}')
print(f'MAE on test set: {mean_absolute_error(y_test, pred):.2f} mg/dl')
feature_names = model.named_steps['preprocess'].get_feature_names_out()
coefs = pd.Series(model.named_steps['regression'].coef_, index=feature_names).sort_values(key=np.abs, ascending=False)
print('\nTop three absolute coefficients:')
print(coefs.head(3).rename(lambda x: x.split('__')[-1]).round(2))

Train/test split: 80/20 with random_state=42
R² on test set: -0.096
MAE on test set: 41.64 mg/dl

Top three absolute coefficients:
sex_1: -31.68
thal_6: -29.92
age_risk_group_65+: -17.82


## Phase 4 coefficient interpretation and limitations
The largest coefficient is for `sex_1`, which is negative, meaning that after holding the other encoded and scaled predictors constant, male patients are predicted to have lower cholesterol than the reference group of female patients in this sample. The `thal_6` coefficient is also negative, so patients coded with fixed-defect thalassemia have lower predicted cholesterol than the reference thalassemia category, but this should not be interpreted causally because thalassemia status and cholesterol are not mechanistically isolated in this observational dataset. The `age_risk_group_65+` coefficient is negative relative to the youngest reference group after accounting for other model inputs, suggesting that the oldest group does not necessarily have the highest predicted cholesterol once sex, disease indicators, exercise measurements, and other covariates are included. The test R² is slightly below zero, so the model performs worse than simply predicting the test-set mean cholesterol and has limited predictive usefulness. The MAE of about 42 mg/dl is clinically meaningful error, which reinforces that cholesterol is only weakly explained by this feature set. Overall, the coefficients are best treated as descriptive linear associations in a small, old clinical sample rather than stable clinical rules.
